In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Module_01_User_and_Body_Profile.md").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display, Markdown


# 03 — PaddleOCR + rules baseline

**Research question:** OCR local → ghép hàng → parser có đọc đúng đơn vị, số thập phân và missing trên smoke fixtures không?

Dataset: synthetic v1; test theo subject/phiếu gốc. Baseline: PP-OCRv5 mobile CPU, PaddleOCR 3.7.0 + parser rules-0.1.0. Không dùng transcript làm đầu vào khi đo OCR.

In [2]:
config = json.loads((ROOT / "configs/inbody_ocr_v5_historical.json").read_text(encoding="utf-8"))
display(config)
RERUN_INFERENCE = False  # Set True explicitly for a new measured inference run.
metrics_path = ROOT / "experiments/inbody_paddle_smoke/metrics.json"
if RERUN_INFERENCE or not metrics_path.exists():
    from scripts.run_inbody_ocr import run_benchmark
    run_benchmark(config_path=ROOT / "configs/inbody_ocr_v5_historical.json")
    metrics_path = ROOT / "experiments/inbody_ocr_smoke_v1/metrics.json"
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
assert metrics["engine"] == "paddle" and metrics["synthetic_only"]
assert metrics["config"] == config, "Cached config differs: rerun inference"
import hashlib
assert metrics["manifest_sha256"] == hashlib.sha256((ROOT / config["manifest"]).read_bytes()).hexdigest()
print("Measured packages:", metrics["package_versions"])
print({key: metrics[key] for key in ["documents", "field_f1", "document_success_rate", "cer", "wer", "mean_latency_seconds"]})

{'experiment_id': 'inbody_ocr_smoke_v1',
 'manifest': 'data/manifests/inbody_synthetic_v1.json',
 'split': 'smoke_test',
 'confidence_threshold': 0.85,
 'device': 'cpu',
 'detection_model': 'PP-OCRv5_mobile_det',
 'recognition_model': 'en_PP-OCRv5_mobile_rec',
 'note': 'English simple-row smoke fixtures only. Vietnamese and real InBody layouts are not evaluated.'}

Measured packages: {'paddleocr': '3.7.0', 'paddlepaddle': '3.3.1', 'paddlex': '3.7.2', 'Pillow': '12.3.0'}
{'documents': 8, 'field_f1': 1.0, 'document_success_rate': 1.0, 'cer': 0.0003824091778202677, 'wer': 0.004597701149425287, 'mean_latency_seconds': 4.421457387499231}


In [3]:
display(Markdown((metrics_path.parent / "report.md").read_text(encoding="utf-8")))
predictions = json.loads((metrics_path.parent / "predictions.json").read_text(encoding="utf-8"))
for prediction in predictions:
    if prediction["quality_flags"]:
        print(prediction["document_id"], prediction["quality_flags"])

# InBody paddle smoke result

**Synthetic fixtures only — real InBody accuracy is unknown.**

Documents: 8; split: smoke_test; real documents: 0.
Field F1: 1.0; document success: 1.0.
CER: 0.0003824091778202677; WER: 0.004597701149425287.

| Field | Unit | Labeled | Paired | MAE | Exact |
| --- | --- | ---: | ---: | ---: | ---: |
| weight_kg | kg | 8 | 8 | 0.0 | 8 |
| height_cm | cm | 8 | 8 | 0.0 | 8 |
| skeletal_muscle_mass_kg | kg | 8 | 8 | 0.0 | 8 |
| body_fat_mass_kg | kg | 8 | 8 | 0.0 | 8 |
| percent_body_fat | % | 8 | 8 | 0.0 | 8 |
| total_body_water_l | L | 7 | 7 | 0.0 | 7 |
| ecw_tbw_ratio | ratio | 8 | 8 | 0.0 | 8 |
| basal_metabolic_rate_kcal | kcal/day | 8 | 8 | 0.0 | 8 |
| bmi | kg/m2 | 8 | 8 | 0.0 | 8 |

## Error analysis

- OCR text synthetic-009: 1 character edits; [{'reference': ' ', 'prediction': ''}]

Only simple English rows, one original renderer, clean scans and decimal dot/comma are covered.
No real sheets, phone-photo distortions, Vietnamese OCR or multi-column/segmental layouts tested.
Oracle mode bypasses OCR and must not be reported as OCR accuracy. Bounds are engineering checks, not clinical advice.


synthetic-015 ['total_body_water_l:missing']


## Error analysis / conclusion

MAE tính trên cặp có cả reference và prediction; phải đọc cùng coverage để tránh bỏ qua missing. Null/null không tăng field F1. CER/WER chuẩn hóa NFC, casefold và khoảng trắng, không bỏ dấu câu/đơn vị. Latency bỏ thời gian nạp engine/tải model. `oracle` chỉ kiểm parser và không có CER/WER.

Kết quả chỉ mô tả 8 ảnh sạch cùng renderer. Chưa có real-world benchmark, correction-rate study, layout-aware/VLM comparison hoặc ngưỡng phát hành. Mọi extraction vẫn cần người dùng kiểm tra.